# Spotify Lyrics Pre-processing

### Goal

Create reusable **text features** for lyrics (Model B) without persisting a combined Spotify+lyrics table.

### Input
- `data/cleaned/lyrics_cleaned.csv`

### Output
- `data/processed/lyrics_tfidf.joblib`

Notes:
- TF-IDF is fit **only on the lyrics corpus** (no labels), so it’s safe.

### 1. Setup & Data Loading

#### 1.1 Path Configuration

In [2]:
import os 

IS_KAGGLE = os.path.exists("/kaggle/input")

if IS_KAGGLE:
    DATA_PATH = "/kaggle/working/lyrics_cleaned.csv"
    OUTPUT_PATH = "/kaggle/working/"
else:
    DATA_PATH = "../data/cleaned/lyrics_cleaned.csv"
    OUTPUT_PATH = "../data/processed/"

# Create output directory if it doesn't exist
os.makedirs(OUTPUT_PATH, exist_ok=True)

# Print paths for confirmation
print(f"IS_KAGGLE: {IS_KAGGLE}")
print(f"DATA_PATH: {DATA_PATH}")
print(f"OUTPUT_PATH: {OUTPUT_PATH}")

IS_KAGGLE: False
DATA_PATH: ../data/cleaned/lyrics_cleaned.csv
OUTPUT_PATH: ../data/processed/


#### 1.2 Import required packages

In [3]:
from pathlib import Path

import joblib
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

#### 1.3 Data Loading

In [5]:
Path(DATA_PATH).exists(), Path(DATA_PATH)

(True, PosixPath('../data/cleaned/lyrics_cleaned.csv'))

In [6]:
lyr = pd.read_csv(DATA_PATH)

In [7]:
lyr.shape, lyr.columns.tolist()

((57438, 8),
 ['artist',
  'song',
  'link',
  'text',
  'artist_norm',
  'title_norm',
  'lyrics',
  'lyrics_char_len'])

### 2. Lyrics Processing (Vectorize Lyrics)

Fits a reusable TF‑IDF vectorizer on the lyrics corpus

#### 2.1 Build TfidVectorizer

We build a TF‑IDF text representation of the lyrics dataset so that we can utilise the numeric features created from it.

- **Creates a** `TfidfVectorizer` (`vec`) that will convert text → numeric features.  

    1. `analyzer="char_wb"`: uses character **n‑grams** within word boundaries (so it’s robust to spelling/punctuation differences and small variations like “remaster” vs “remastered”).  
    
    2. `ngram_range=(3, 5)`: extracts 3-, 4-, and 5-character chunks (e.g., `"love"` contributes `lov`, `ove`, etc.).  
    
    3. `min_df=5`: ignores n‑grams that appear in fewer than 5 documents (removes extremely rare noise).  
 
    4. `max_df=0.5`: ignores n‑grams that appear in more than 50% of documents (removes overly common, low-signal patterns).  
&nbsp;  
- `fit_transform(...)`:

    1. fits the vectorizer’s vocabulary + IDF weights on all lyrics, then
    
    2. transforms each lyric into a sparse TF‑IDF vector.
    
    3. `fillna("")` ensures missing lyrics become empty strings instead of errors.  
&nbsp;  
- `X.shape` shows the resulting matrix size:
    
    1. rows = number of lyric documents
    
    2. columns = number of retained character n‑gram features

In [9]:
vec = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 5),
    min_df=5,
    max_df=0.5,
)


In [10]:
X = vec.fit_transform(lyr["lyrics"].fillna(""))

### 3. Check

In [11]:
X.shape

(57438, 128303)

### 4. Output

We dump vec (the fitted TfidfVectorizer) because it contains the feature definition needed to reproduce X consistently later:  

**What vec stores**: the learned vocabulary (which n‑grams map to which column), the IDF weights, and all preprocessing settings (`analyzer`, `ngram_range`, `min_df`, `max_df`, etc.).  

Without the same vec, the columns of X won’t match across runs/datasets.  

**Why not dump X**:

`X` is just the transformed matrix for this specific lyrics corpus. It’s not reusable for new data unless you also have the exact mapping.  

In Model B, you’ll transform lyrics for the matched Spotify subset (and later any new lyrics) using the same `vec`:  

  - `X_subset` = `vec.transform(subset_lyrics)`  

`X` can be very large and is easy to recompute once vec is saved.  

So: save `vec` to guarantee consistent features, and recompute `X` (or subsets of it) whenever needed.

In [12]:
os.makedirs(OUTPUT_PATH, exist_ok=True)
VEC_OUT = "../data/processed/lyrics_tfidf.joblib"

In [13]:
joblib.dump(vec, VEC_OUT)
print(f"Saved vectorizer to: {VEC_OUT}")

Saved vectorizer to: ../data/processed/lyrics_tfidf.joblib
